In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Read CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')

# Normalize column names to a common 'kpi_name' and add source_file
df_metric = df_metric.rename(columns={'kpi_name': 'kpi_name'})
df_metric['source_file'] = 'metric.csv'

df_trace = df_trace.rename(columns={'trace_name': 'kpi_name'})
df_trace['source_file'] = 'trace.csv'

df_log = df_log.rename(columns={'log_name': 'kpi_name'})
df_log['source_file'] = 'log.csv'

# Convert timestamp to timezone-aware datetime in Asia/Shanghai (UTC+8)
for df in (df_metric, df_trace, df_log):
    df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # ensure numeric values
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Aggregation function
def summarize_df(df, source_name):
    grp = df.groupby(['cmdb_id', 'kpi_name'], as_index=False)
    summary = grp.agg(
        sample_count = ('value', 'count'),
        min_timestamp = ('ts_dt', 'min'),
        max_timestamp = ('ts_dt', 'max'),
        p50 = ('value', lambda s: s.quantile(0.50)),
        p90 = ('value', lambda s: s.quantile(0.90)),
        p95 = ('value', lambda s: s.quantile(0.95)),
        p99 = ('value', lambda s: s.quantile(0.99)),
    )
    summary['source_file'] = source_name
    # Reorder columns as requested
    summary = summary[['source_file','cmdb_id','kpi_name','sample_count','min_timestamp','max_timestamp','p50','p90','p95','p99']]
    # sort by sample_count desc and keep top 20
    summary = summary.sort_values(by='sample_count', ascending=False).head(20).reset_index(drop=True)
    return summary

metric_summary = summarize_df(df_metric, 'metric.csv')
trace_summary = summarize_df(df_trace, 'trace.csv')
log_summary = summarize_df(df_log, 'log.csv')

# Deduplicated sorted list of all cmdb_id values across the three files
all_cmdb_ids = pd.Series(
    sorted(
        set(df_metric['cmdb_id'].dropna().unique()).union(
            set(df_trace['cmdb_id'].dropna().unique()),
            set(df_log['cmdb_id'].dropna().unique())
        )
    ),
    name='cmdb_id'
)

# Display compact results (up to top 20 per source_file)
metric_summary, trace_summary, log_summary, all_cmdb_ids

```
Out[1]:
```
summary = (
    "Summary of telemetry aggregations (based on the computed group summaries):\n\n"
    "1) metric.csv (top groups):\n"
    "- Top groups returned have sample_count = 31, covering 2021-03-23 18:00 to 2021-03-23 18:30 (Asia/Shanghai, UTC+8).\n"
    "- Example top KPIs: IG01 JVM-Threads (JVM_ThreadCount), Mysql02 GetConnectedStateOfMysqld and GetResponseTimeOfMysqld.\n"
    "- Percentiles (p50/p90/p95/p99) were computed on the full series for each KPI.\n\n"
    "2) trace.csv (top groups):\n"
    "- Top groups returned also have sample_count = 31, covering 2021-03-23 18:00 to 2021-03-23 18:30 (Asia/Shanghai).\n"
    "- Example top KPIs: dockerB2 trace.self.row_count, dockerA2 trace.self.duration_mean, trace.self.duration_p95.\n"
    "- Percentiles (p50/p90/p95/p99) reflect full-series values per trace KPI.\n\n"
    "3) log.csv (top groups):\n"
    "- Many top log groups have sample_count = 31 and the same 18:00–18:30 time range; some groups (e.g., Mysql01) have 29 samples, and GC metrics show fewer samples (12–14).\n"
    "- Example top KPIs: Tomcat01 log.row_count and log.error_count, apache01/apache02 log.row_count and log.error_count.\n\n"
    "4) Cross-file inventory:\n"
    "- A deduplicated, sorted set of cmdb_id values across the three files contains 29 unique cmdb_id entries.\n\n"
    "Notes:\n"
    "- All percentile thresholds were computed on each KPI's full available series before any further filtering.\n"
    "- Output is a compact aggregation per (cmdb_id, KPI) with sample counts, time ranges, and p50/p90/p95/p99 values."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   source_file   cmdb_id                                           kpi_name  sample_count             min_timestamp             max_timestamp           p50           p90           p95           p99
0   metric.csv      IG01           JVM-Threads_7778_JVM_ThreadCount_Threads            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  7.100000e+01  7.500000e+01  7.550000e+01  7.600000e+01
1   metric.csv   Mysql02         Mysql-MySQL_3306_GetConnectedStateOfMysqld            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
2   metric.csv   Mysql02           Mysql-MySQL_3306_GetResponseTimeOfMysqld            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
3   metric.csv  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
4   metric.csv  Tomcat03                 OSLinux-OSLinux_ZABBIX_Host_Uptime            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  7.693430e+06  7.694152e+06  7.694243e+06  7.694316e+06
5   metric.csv  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  6.210000e+02  6.210000e+02  6.210000e+02  6.210000e+02
6   metric.csv  Tomcat03         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
7   metric.csv  Tomcat03      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
8   metric.csv  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
9   metric.csv  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  4.016560e+05  4.016560e+05  4.016560e+05  4.016560e+05
10  metric.csv  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  1.710823e+09  1.711720e+09  1.711804e+09  1.711859e+09
11  metric.csv  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  2.822467e+06  2.825575e+06  2.825947e+06  2.826234e+06
12  metric.csv  Tomcat03    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
13  metric.csv  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
14  metric.csv  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
15  metric.csv  Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
16  metric.csv  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02
17  metric.csv  Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
18  metric.csv  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
19  metric.csv  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00,    source_file   cmdb_id                           kpi_name  sample_count             min_timestamp             max_timestamp          p50          p90          p95          p99
0    trace.csv  dockerB2               trace.self.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00   784.000000  1140.000000  1430.500000  1498.300000
1    trace.csv  dockerA2           trace.self.duration_mean            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.011198     0.016764     0.017370     0.018860
2    trace.csv  dockerA2            trace.self.duration_p95            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.007000     0.009000     0.009000     0.009000
3    trace.csv  dockerA2               trace.self.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  2192.000000  2738.000000  2894.000000  3078.200000
4    trace.csv  dockerA2        trace.to_MG01.duration_mean            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.038028     0.152417     0.186609     0.607870
5    trace.csv  dockerA2         trace.to_MG01.duration_p95            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.040900     1.018100     1.020525     2.549250
6    trace.csv  dockerA2            trace.to_MG01.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00    32.000000    53.000000    70.500000    85.200000
7    trace.csv  dockerA2      trace.from_MG01.duration_mean            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.085152     0.151908     0.174802     0.182271
8    trace.csv  dockerA2       trace.from_MG01.duration_p95            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.705800     1.028600     1.029000     1.046780
9    trace.csv  dockerA2          trace.from_MG01.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00   142.000000   190.000000   199.000000   209.700000
10   trace.csv      MG01           trace.self.duration_mean            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.111682     0.201276     0.224469     0.339518
11   trace.csv      MG01            trace.self.duration_p95            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     1.026000     1.081750     1.219800     2.232155
12   trace.csv      MG01  trace.from_dockerA2.duration_mean            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.038028     0.152417     0.186609     0.607870
13   trace.csv      MG01   trace.from_dockerA2.duration_p95            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.040900     1.018100     1.020525     2.549250
14   trace.csv      MG01      trace.from_dockerA2.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00    32.000000    53.000000    70.500000    85.200000
15   trace.csv      MG01  trace.from_dockerB1.duration_mean            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.039566     0.075794     0.098469     0.203700
16   trace.csv      MG01   trace.from_dockerB1.duration_p95            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.070100     0.791000     0.963600     1.485540
17   trace.csv      MG01      trace.from_dockerB1.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00    99.000000   190.000000   261.000000   331.700000
18   trace.csv      MG01      trace.from_Tomcat02.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00   128.000000   161.000000   170.500000   178.600000
19   trace.csv      MG01  trace.from_Tomcat03.duration_mean            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00     0.135000     0.245287     0.300841     0.361893,    source_file   cmdb_id          kpi_name  sample_count             min_timestamp             max_timestamp      p50      p90       p95       p99
0      log.csv  Tomcat01     log.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  250.000  316.000  332.5000  337.4000
1      log.csv  Tomcat01   log.error_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00    0.000    0.000    0.0000    0.0000
2      log.csv  apache02   log.error_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00    0.000    0.000    0.0000    0.0000
3      log.csv  apache01     log.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  524.000  625.000  653.5000  673.6000
4      log.csv  Tomcat02   log.error_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00    0.000    0.000    0.0000    0.0000
5      log.csv  Tomcat02     log.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  266.000  312.000  320.5000  340.8000
6      log.csv  Tomcat03   log.error_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00    0.000    0.000    0.0000    0.0000
7      log.csv  Tomcat04     log.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  267.000  314.000  325.5000  334.6000
8      log.csv  Tomcat04   log.error_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00    0.000    0.000    0.0000    0.0000
9      log.csv  Tomcat03     log.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  261.000  327.000  343.5000  361.6000
10     log.csv  apache02     log.row_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00  525.000  627.000  647.0000  678.9000
11     log.csv  apache01   log.error_count            31 2021-03-23 18:00:00+08:00 2021-03-23 18:30:00+08:00    0.000    0.000    0.0000    0.0000
12     log.csv   Mysql01     log.row_count            29 2021-03-23 18:00:00+08:00 2021-03-23 18:29:00+08:00    4.000    8.200    9.6000   14.3200
13     log.csv   Mysql01   log.error_count            29 2021-03-23 18:00:00+08:00 2021-03-23 18:29:00+08:00    0.000    0.000    0.0000    0.0000
14     log.csv  Tomcat04  log.gc_pause_max            14 2021-03-23 18:00:00+08:00 2021-03-23 18:27:00+08:00    0.075    0.090    0.0900    0.0900
15     log.csv  Tomcat03  log.gc_pause_max            12 2021-03-23 18:01:00+08:00 2021-03-23 18:27:00+08:00    0.060    0.080    0.0980    0.1156
16     log.csv  Tomcat01  log.gc_pause_max            12 2021-03-23 18:01:00+08:00 2021-03-23 18:28:00+08:00    0.065    0.080    0.0800    0.0800
17     log.csv  Tomcat02  log.gc_pause_max            12 2021-03-23 18:01:00+08:00 2021-03-23 18:28:00+08:00    0.060    0.089    0.0945    0.0989, 0         IG01
1         IG02
2         MG01
3         MG02
4      Mysql01
        ...   
24    apache02
25    dockerA1
26    dockerA2
27    dockerB1
28    dockerB2
Name: cmdb_id, Length: 29, dtype: object)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load metric data (reuse if already in memory is okay, but read to ensure fresh state)
df_metric = pd.read_csv('metric.csv')

# Normalize columns
if 'kpi_name' not in df_metric.columns and 'kpi' in df_metric.columns:
    df_metric = df_metric.rename(columns={'kpi': 'kpi_name'})

df_metric = df_metric.rename(columns={'kpi_name': 'kpi_name'})
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Convert timestamp to timezone-aware Asia/Shanghai datetimes
df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Group keys
keys = ['cmdb_id', 'kpi_name']

# 1) Compute global P95 per group on full series
thresholds_p95 = df_metric.groupby(keys, as_index=False)['value'].quantile(0.95).rename(columns={'value': 'global_p95'})

# Incident window (inclusive) in Asia/Shanghai
start = tz.localize(pd.to_datetime('2021-03-23 18:00:00'))
end = tz.localize(pd.to_datetime('2021-03-23 18:30:00'))

# 2) Filter points to the incident window
window_df = df_metric[(df_metric['ts_dt'] >= start) & (df_metric['ts_dt'] <= end)].copy()

# 3) For groups with at least one point in the window compute window stats
if not window_df.empty:
    window_stats = window_df.groupby(keys, as_index=False).agg(
        window_sample_count = ('value', 'count'),
        window_min_timestamp = ('ts_dt', 'min'),
        window_max_timestamp = ('ts_dt', 'max'),
        window_mean = ('value', 'mean'),
        window_min_value = ('value', 'min'),
        window_max_value = ('value', 'max'),
    )
else:
    window_stats = pd.DataFrame(columns=keys + [
        'window_sample_count','window_min_timestamp','window_max_timestamp',
        'window_mean','window_min_value','window_max_value'
    ])

# Merge global p95 into window stats
merged = window_stats.merge(thresholds_p95, on=keys, how='left')

# Compute anomalies: attach global_p95 to each window point then count where value > global_p95
window_with_thr = window_df.merge(thresholds_p95, on=keys, how='left')
# Condition strictly greater than global_p95
anom_points = window_with_thr[window_with_thr['value'] > window_with_thr['global_p95']].copy()

if not anom_points.empty:
    anom_agg = anom_points.groupby(keys, as_index=False).agg(
        count_anomalies = ('value','count'),
        earliest_anomaly_timestamp = ('ts_dt','min')
    )
else:
    # empty frame with the expected columns
    anom_agg = pd.DataFrame(columns=keys + ['count_anomalies','earliest_anomaly_timestamp'])

# Merge anomaly aggregates into merged stats
merged = merged.merge(anom_agg, on=keys, how='left')
merged['count_anomalies'] = merged['count_anomalies'].fillna(0).astype(int)
# earliest_anomaly_timestamp may be NaT; keep as datetime
merged['earliest_anomaly_timestamp'] = pd.to_datetime(merged['earliest_anomaly_timestamp']).dt.tz_localize(None)
# The merged window_min/max timestamps are timezone-aware; keep them as-is. For compactness we'll present earliest anomaly in Asia/Shanghai if present.
merged['earliest_anomaly_timestamp'] = merged['earliest_anomaly_timestamp'].apply(
    lambda x: tz.localize(x) if pd.notna(x) else pd.NaT
)

# 4) Keep only groups where count_anomalies > 0, sort and limit to top 20
result_p95 = merged[merged['count_anomalies'] > 0].copy()
result_p95 = result_p95.sort_values(by='count_anomalies', ascending=False).head(20)

# Select and order columns as requested
cols = ['cmdb_id','kpi_name','global_p95','window_sample_count','count_anomalies',
        'earliest_anomaly_timestamp','window_min_value','window_max_value','window_mean']
result_p95 = result_p95[cols]

# If no groups breach global P95, repeat using global P90
if result_p95.shape[0] == 0:
    thresholds_p90 = df_metric.groupby(keys, as_index=False)['value'].quantile(0.90).rename(columns={'value': 'global_p90'})
    # Merge p90 into window stats
    merged90 = window_stats.merge(thresholds_p90, on=keys, how='left')
    window_with_thr90 = window_df.merge(thresholds_p90, on=keys, how='left')
    anom_points90 = window_with_thr90[window_with_thr90['value'] > window_with_thr90['global_p90']].copy()
    if not anom_points90.empty:
        anom_agg90 = anom_points90.groupby(keys, as_index=False).agg(
            count_anomalies = ('value','count'),
            earliest_anomaly_timestamp = ('ts_dt','min')
        )
    else:
        anom_agg90 = pd.DataFrame(columns=keys + ['count_anomalies','earliest_anomaly_timestamp'])
    merged90 = merged90.merge(anom_agg90, on=keys, how='left')
    merged90['count_anomalies'] = merged90['count_anomalies'].fillna(0).astype(int)
    merged90['earliest_anomaly_timestamp'] = pd.to_datetime(merged90['earliest_anomaly_timestamp']).dt.tz_localize(None)
    merged90['earliest_anomaly_timestamp'] = merged90['earliest_anomaly_timestamp'].apply(
        lambda x: tz.localize(x) if pd.notna(x) else pd.NaT
    )
    result_p90 = merged90[merged90['count_anomalies'] > 0].copy()
    result_p90 = result_p90.sort_values(by='count_anomalies', ascending=False).head(20)
    cols90 = ['cmdb_id','kpi_name','global_p90','window_sample_count','count_anomalies',
             'earliest_anomaly_timestamp','window_min_value','window_max_value','window_mean']
    # Ensure columns exist; if none, return empty DataFrame with expected columns
    if result_p90.shape[0] == 0:
        result = pd.DataFrame(columns=cols90)
    else:
        result = result_p90[cols90]
else:
    result = result_p95

# Final compact result (top 20). Display as the variable 'result'
result

```
Out[2]:
```
Summary (metric.csv, global P95, window 2021-03-23 18:00–18:30 Asia/Shanghai):

- Several (multiple) (cmdb_id, kpi) groups had values in the incident window exceeding their global P95. Each reported group had 2 anomalous points in-window (count_anomalies = 2).
- Examples / top candidates (cmdb_id — kpi_name — global_p95 — earliest_anomaly — window_sample_count — window_min / window_max / mean):
  - IG02 — JVM-Memory_7778_JVM_Memory_HeapMemoryUsed — 8.537806e+08 — 2021-03-23 18:13:00+08:00 — 31 — 1.025327e+08 / 8.878561e+08 / 4.946429e+08
  - Tomcat04 — OSLinux-OSLinux_ZABBIX_Host_Uptime — 7.694207e+06 — 2021-03-23 18:29:00+08:00 — 31 — 7.692492e+06 / 7.694298e+06 / 7.693394e+06
  - Tomcat04 — Tomcat-Requests_7441_..."_Processing..." — 9.806206e+08 — 2021-03-23 18:29:00+08:00 — 31 — 9.784035e+08 / 9.807271e+08 / 9.796656e+08
  - Tomcat04 — Tomcat-Requests_7441_..."_RequestCount..." — 1.894799e+06 — 2021-03-23 18:29:00+08:00 — 31 — 1.887208e+06 / 1.895160e+06 / 1.891364e+06
  - Tomcat04 — Tomcat-Threads_7441_..."_CurrentThreads..." — 4.5 — 2021-03-23 18:03:00+08:00 — 31 — 0.0 / 5.0 / 1.2258
  - Tomcat03 — OSLinux-CPU_CPU_CPULoad — 1.3415 — 2021-03-23 18:18:00+08:00 — 30 — 1.01 / 1.44 / 1.1267
  - Tomcat03 — various OSLinux CPU metrics, request/process KPIs and host uptime also breached P95 with earliest anomalies around 18:00–18:29.
  - MG01 / MG02 — JVM heap and host-uptime related KPIs breached P95 as well (earliest anomalies at ~18:21 and 18:29).
  - Tomcat02 — host uptime and request/process KPIs breached P95 with earliest anomalies ~18:29.

- Overall observation: JVM heap metrics (IG02, MG02), Tomcat request/process/thread KPIs (Tomcat02/03/04), and several OSLinux host/CPU metrics show breaches of their global P95 during the incident window — each group had two points above the group-wide P95.

Suggested next step: investigate those listed JVM and Tomcat KPIs and the corresponding timestamps (earliest anomaly times) to find root causes (e.g., GC/heap growth, load spikes, or process/thread changes).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_p95  window_sample_count  count_anomalies earliest_anomaly_timestamp  window_min_value  window_max_value   window_mean
20        IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.537806e+08                   31                2  2021-03-23 18:13:00+08:00      1.025327e+08      8.878561e+08  4.946429e+08
1110  Tomcat04                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.694207e+06                   31                2  2021-03-23 18:29:00+08:00      7.692492e+06      7.694298e+06  7.693394e+06
1118  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...  9.806206e+08                   31                2  2021-03-23 18:29:00+08:00      9.784035e+08      9.807271e+08  9.796656e+08
1119  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  1.894799e+06                   31                2  2021-03-23 18:29:00+08:00      1.887208e+06      1.895160e+06  1.891364e+06
1133  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  4.500000e+00                   31                2  2021-03-23 18:03:00+08:00      0.000000e+00      5.000000e+00  1.225806e+00
901   Tomcat03                            OSLinux-CPU_CPU_CPULoad  1.341500e+00                   30                2  2021-03-23 18:18:00+08:00      1.010000e+00      1.440000e+00  1.126667e+00
902   Tomcat03                         OSLinux-CPU_CPU_CPUSysTime  6.428033e-01                   30                2  2021-03-23 18:00:00+08:00      4.324320e-01      6.946000e-01  5.588130e-01
903   Tomcat03                        OSLinux-CPU_CPU_CPUUserTime  2.586871e+01                   30                2  2021-03-23 18:00:00+08:00      2.546785e+01      2.604825e+01  2.566123e+01
904   Tomcat03                             OSLinux-CPU_CPU_CPUWio  4.899359e-01                   30                2  2021-03-23 18:02:00+08:00      1.250000e-02      7.113000e-01  1.451509e-01
139       MG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.695012e+06                   31                2  2021-03-23 18:29:00+08:00      7.693297e+06      7.695105e+06  7.694200e+06
142       MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.412856e+08                   31                2  2021-03-23 18:21:00+08:00      1.525904e+08      9.792490e+08  5.457801e+08
146       MG02           JVM-Threads_7779_JVM_ThreadCount_Threads  7.350000e+01                   31                2  2021-03-23 18:05:00+08:00      6.800000e+01      7.400000e+01  7.090323e+01
989   Tomcat03                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.694243e+06                   31                2  2021-03-23 18:29:00+08:00      7.692527e+06      7.694334e+06  7.693430e+06
997   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.711804e+09                   31                2  2021-03-23 18:29:00+08:00      1.709608e+09      1.711872e+09  1.710847e+09
998   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.825947e+06                   31                2  2021-03-23 18:29:00+08:00      2.818163e+06      2.826309e+06  2.822442e+06
905   Tomcat03                        OSLinux-CPU_CPU_CPUidleutil  7.372344e+01                   30                2  2021-03-23 18:24:00+08:00      7.239600e+01      7.375052e+01  7.330529e+01
968   Tomcat03       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  5.000000e-01                   31                2  2021-03-23 18:03:00+08:00      0.000000e+00      1.000000e+00  6.451613e-02
867   Tomcat02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.694247e+06                   31                2  2021-03-23 18:29:00+08:00      7.692530e+06      7.694338e+06  7.693434e+06
875   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.722621e+09                   31                2  2021-03-23 18:29:00+08:00      1.720409e+09      1.722703e+09  1.721656e+09
876   Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.835772e+06                   31                2  2021-03-23 18:29:00+08:00      2.828271e+06      2.836128e+06  2.832366e+06```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load trace data (reuse if present)
df_trace = pd.read_csv('trace.csv')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Candidate cmdb_ids
candidates = ['IG02','MG01','MG02','Tomcat02','Tomcat03','Tomcat04']

# Incident window (inclusive) in Asia/Shanghai
start = tz.localize(pd.to_datetime('2021-03-23 18:00:00'))
end = tz.localize(pd.to_datetime('2021-03-23 18:30:00'))

keys = ['cmdb_id','trace_name']

# Restrict to candidate cmdb_ids for efficiency
df_cand = df_trace[df_trace['cmdb_id'].isin(candidates)].copy()

# Compute global P95 per group on full series (rule: compute before filtering)
thr_p95 = df_cand.groupby(keys, as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# Filter to incident window
window_df = df_cand[(df_cand['ts_dt'] >= start) & (df_cand['ts_dt'] <= end)].copy()

# Compute window stats per group (only groups with at least one point in window)
if not window_df.empty:
    window_stats = window_df.groupby(keys, as_index=False).agg(
        window_sample_count = ('value','count'),
        window_min_timestamp = ('ts_dt','min'),
        window_max_timestamp = ('ts_dt','max'),
        window_min_value = ('value','min'),
        window_max_value = ('value','max'),
        window_mean = ('value','mean')
    )
else:
    window_stats = pd.DataFrame(columns=keys + [
        'window_sample_count','window_min_timestamp','window_max_timestamp',
        'window_min_value','window_max_value','window_mean'
    ])

# Merge global P95 into window stats
merged = window_stats.merge(thr_p95, on=keys, how='left')

# Find anomaly points: value > global_p95 (use merged thresholds)
window_with_thr = window_df.merge(thr_p95, on=keys, how='left')
anom_points = window_with_thr[window_with_thr['value'] > window_with_thr['global_p95']].copy()

if not anom_points.empty:
    anom_agg = anom_points.groupby(keys, as_index=False).agg(
        count_anomalies = ('value','count'),
        earliest_anomaly_timestamp = ('ts_dt','min')
    )
else:
    anom_agg = pd.DataFrame(columns=keys + ['count_anomalies','earliest_anomaly_timestamp'])

# Combine and prepare final result for P95
merged = merged.merge(anom_agg, on=keys, how='left')
merged['count_anomalies'] = merged['count_anomalies'].fillna(0).astype(int)

# Keep only groups with anomalies > 0
result_p95 = merged[merged['count_anomalies'] > 0].copy()

# Select and order columns as requested
cols_out = ['cmdb_id','trace_name','global_p95','window_sample_count','count_anomalies',
            'earliest_anomaly_timestamp','window_min_value','window_max_value','window_mean']

if not result_p95.empty:
    # Sort by earliest anomaly timestamp ascending and limit to top 20
    result_p95 = result_p95.sort_values(by='earliest_anomaly_timestamp', ascending=True).head(20)
    result = result_p95[cols_out]
else:
    # If no breaches at P95, repeat with P90
    thr_p90 = df_cand.groupby(keys, as_index=False)['value'].quantile(0.90).rename(columns={'value':'global_p90'})
    merged90 = window_stats.merge(thr_p90, on=keys, how='left')
    window_with_thr90 = window_df.merge(thr_p90, on=keys, how='left')
    anom_points90 = window_with_thr90[window_with_thr90['value'] > window_with_thr90['global_p90']].copy()
    if not anom_points90.empty:
        anom_agg90 = anom_points90.groupby(keys, as_index=False).agg(
            count_anomalies = ('value','count'),
            earliest_anomaly_timestamp = ('ts_dt','min')
        )
    else:
        anom_agg90 = pd.DataFrame(columns=keys + ['count_anomalies','earliest_anomaly_timestamp'])
    merged90 = merged90.merge(anom_agg90, on=keys, how='left')
    merged90['count_anomalies'] = merged90['count_anomalies'].fillna(0).astype(int)
    result_p90 = merged90[merged90['count_anomalies'] > 0].copy()
    if not result_p90.empty:
        # rename global_p90 to global_p95 column name for consistent output header
        result_p90 = result_p90.rename(columns={'global_p90':'global_p95'})
        result_p90 = result_p90.sort_values(by='earliest_anomaly_timestamp', ascending=True).head(20)
        result = result_p90[cols_out]
    else:
        # No anomalies at P95 or P90: return empty DataFrame with expected columns
        result = pd.DataFrame(columns=cols_out)

# Final compact result
result

```
Out[3]:
```
summary = (
    "Summary (trace.csv; candidates IG02, MG01, MG02, Tomcat02/03/04; incident window 2021-03-23 18:00–18:30 Asia/Shanghai):\n\n"
    "- Multiple trace series breached their group-wide global P95 during the incident window. Every reported group had 2 anomalous points in-window.\n"
    "- Earliest anomalies occur immediately at or shortly after 18:00 and cluster in the 18:00–18:03 range (sorted by earliest anomaly).\n\n"
    "- Notable breach patterns:\n"
    "  • High row_count spikes on flows involving IG02: trace.to_Tomcat02.row_count, trace.to_Tomcat03.row_count, trace.to_Tomcat04.row_count, and trace.from_IG02.row_count (affecting Tomcat02/03/04) — earliest anomalies at ~18:00–18:01.\n"
    "  • Tomcat and MG endpoints show elevated row_count metrics: trace.to_MG01.row_count, trace.from_Tomcat03.row_count, trace.from_IG02.row_count, and trace.self.row_count for Tomcat03/Tomcat04 — all with 2 anomalies and earliest times ~18:00–18:01.\n"
    "  • MG02 shows latency-related breaches: duration_mean and duration_p95 series (trace.from_dockerA1.duration_mean/p95, trace.from_dockerA2.duration_p95, trace.from_dockerB1.duration_mean/p95) — indicating transient latency spikes starting ~18:00–18:01.\n"
    "  • Additional MG02 row_count flows (to/from dockerA1/A2, to Tomcat01) also exceeded their P95 around 18:03.\n"
    "\n"
    "- Interpretation / next steps:\n"
    "  1) There was a short-lived surge in request/row counts originating from IG02 (seen across Tomcat02/03/04 and MG01), coinciding with latency spikes at MG02.\n"
    "  2) Investigate traces at the listed earliest anomaly timestamps (18:00–18:03) for root cause: high incoming traffic from IG02, queuing or processing delays in MG02, or upstream load imbalance.\n"
    "  3) Correlate these trace breaches with the metric-level anomalies (JVM, CPU, requests) already observed for Tomcat and MG nodes to pinpoint whether load, JVM/memory, or network/processing latency is primary.\n\n"
    "In short: short, simultaneous spikes in row_count (traffic) from IG02 and latency spikes at MG02 around 18:00–18:03 are the top trace-level leads to investigate."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name   global_p95  window_sample_count  count_anomalies earliest_anomaly_timestamp  window_min_value  window_max_value  window_mean
125  Tomcat03          trace.from_IG02.row_count   173.550000                   30                2  2021-03-23 18:00:00+08:00         96.000000        186.000000   135.000000
73       MG02   trace.from_dockerA1.duration_p95     0.559960                   30                2  2021-03-23 18:00:00+08:00          0.009000          1.013150     0.126420
134  Tomcat03            trace.to_MG01.row_count   167.500000                   31                2  2021-03-23 18:00:00+08:00          2.000000        197.000000   118.032258
72       MG02  trace.from_dockerA1.duration_mean     0.091198                   30                2  2021-03-23 18:00:00+08:00          0.006462          0.124111     0.028563
76       MG02   trace.from_dockerA2.duration_p95     1.015923                   30                2  2021-03-23 18:00:00+08:00          0.010000          1.035300     0.240057
14       IG02        trace.to_Tomcat03.row_count   173.550000                   30                2  2021-03-23 18:00:00+08:00         96.000000        186.000000   135.000000
26       MG01      trace.from_Tomcat03.row_count   167.500000                   31                2  2021-03-23 18:00:00+08:00          2.000000        197.000000   118.032258
17       IG02        trace.to_Tomcat04.row_count   167.000000                   31                2  2021-03-23 18:01:00+08:00          4.000000        173.000000   127.258065
107  Tomcat02          trace.from_IG02.row_count   164.000000                   31                2  2021-03-23 18:01:00+08:00          1.000000        169.000000   126.129032
56       MG01        trace.to_dockerB1.row_count   209.500000                   31                2  2021-03-23 18:01:00+08:00          3.000000        219.000000   148.967742
11       IG02        trace.to_Tomcat02.row_count   164.000000                   31                2  2021-03-23 18:01:00+08:00          1.000000        169.000000   126.129032
143  Tomcat04          trace.from_IG02.row_count   167.000000                   31                2  2021-03-23 18:01:00+08:00          4.000000        173.000000   127.258065
78       MG02  trace.from_dockerB1.duration_mean     0.106462                   30                2  2021-03-23 18:01:00+08:00          0.007780          0.338314     0.050314
5        IG02               trace.self.row_count  1326.000000                   31                2  2021-03-23 18:01:00+08:00         12.000000       1364.000000  1021.903226
149  Tomcat04               trace.self.row_count  2217.500000                   31                2  2021-03-23 18:01:00+08:00         30.000000       2279.000000  1744.354839
79       MG02   trace.from_dockerB1.duration_p95     0.868932                   30                2  2021-03-23 18:01:00+08:00          0.011000          3.056650     0.243052
62       MG02      trace.from_Tomcat01.row_count   223.500000                   31                2  2021-03-23 18:03:00+08:00          1.000000        278.000000   143.451613
92       MG02        trace.to_dockerA1.row_count   100.200000                   30                2  2021-03-23 18:03:00+08:00         29.000000        126.000000    59.600000
95       MG02        trace.to_dockerA2.row_count   305.000000                   30                2  2021-03-23 18:03:00+08:00        102.000000        383.000000   176.200000
112  Tomcat02            trace.self.duration_p95     0.576375                   31                2  2021-03-23 18:03:00+08:00          0.023000          0.702800     0.123840```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse timezone variable if exists, else create
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Candidates
candidates = ['IG02','MG02','MG01','Tomcat02','Tomcat03','Tomcat04']

# Load and prepare log data
df_log = pd.read_csv('log.csv')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
df_log['ts_dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Restrict to candidate cmdb_ids
df_cand = df_log[df_log['cmdb_id'].isin(candidates)].copy()

keys = ['cmdb_id','log_name']

# Incident window (inclusive) in Asia/Shanghai
start = tz.localize(pd.to_datetime('2021-03-23 18:00:00'))
end = tz.localize(pd.to_datetime('2021-03-23 18:30:00'))

# Function to compute anomalies given a threshold dataframe (group, threshold_colname)
def compute_anomalies(df_cand, thr_df, thr_col):
    # windowed points
    window_df = df_cand[(df_cand['ts_dt'] >= start) & (df_cand['ts_dt'] <= end)].copy()
    if window_df.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','log_name', thr_col, 'window_sample_count','count_anomalies',
            'earliest_anomaly_timestamp','window_min_value','window_max_value','window_mean'
        ])
    # window stats per group
    window_stats = window_df.groupby(keys, as_index=False).agg(
        window_sample_count = ('value','count'),
        window_min_timestamp = ('ts_dt','min'),
        window_max_timestamp = ('ts_dt','max'),
        window_min_value = ('value','min'),
        window_max_value = ('value','max'),
        window_mean = ('value','mean')
    )
    # merge threshold
    merged = window_stats.merge(thr_df, on=keys, how='left')
    # attach threshold to each window point to find anomalies
    window_with_thr = window_df.merge(thr_df, on=keys, how='left')
    anom_points = window_with_thr[window_with_thr['value'] > window_with_thr[thr_col]].copy()
    if not anom_points.empty:
        anom_agg = anom_points.groupby(keys, as_index=False).agg(
            count_anomalies = ('value','count'),
            earliest_anomaly_timestamp = ('ts_dt','min')
        )
    else:
        anom_agg = pd.DataFrame(columns=keys + ['count_anomalies','earliest_anomaly_timestamp'])
    merged = merged.merge(anom_agg, on=keys, how='left')
    merged['count_anomalies'] = merged['count_anomalies'].fillna(0).astype(int)
    # Keep only groups with anomalies > 0
    result = merged[merged['count_anomalies'] > 0].copy()
    # Ensure earliest_anomaly_timestamp is timezone-aware (it comes from ts_dt so should be)
    # Select and rename threshold column to a consistent name for output
    result = result.rename(columns={thr_col: 'global_p95'})
    cols_out = ['cmdb_id','log_name','global_p95','window_sample_count','count_anomalies',
                'earliest_anomaly_timestamp','window_min_value','window_max_value','window_mean']
    # sort by earliest anomaly ascending and limit to top 20
    result = result.sort_values(by='earliest_anomaly_timestamp', ascending=True).head(20)
    return result[cols_out]

# 1) Compute global P95 per (cmdb_id, log_name) on full series
thr_p95 = df_cand.groupby(keys, as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# 2-4) Compute anomalies using P95
result_p95 = compute_anomalies(df_cand, thr_p95, 'global_p95')

# If no breaches at P95, repeat with P90
if result_p95.shape[0] == 0:
    thr_p90 = df_cand.groupby(keys, as_index=False)['value'].quantile(0.90).rename(columns={'value':'global_p90'})
    result_p90 = compute_anomalies(df_cand, thr_p90, 'global_p90')
    result = result_p90.copy()
else:
    result = result_p95.copy()

# Prepare deduplicated list of cmdb_id that had log anomalies (if any)
if not result.empty:
    anomalous_cmdbs = pd.Series(sorted(result['cmdb_id'].unique()), name='cmdb_id')
else:
    anomalous_cmdbs = pd.Series([], name='cmdb_id', dtype=object)

# Display compact outputs
result, anomalous_cmdbs

```
Out[4]:
```
summary = (
    "Log analysis (candidate cmdb_ids IG02, MG02, MG01, Tomcat02, Tomcat03, Tomcat04) — incident window 2021-03-23 18:00–18:30 Asia/Shanghai:\n\n"
    "Findings:\n"
    "- Tomcat04: log.row_count exceeded its global P95 (325.5) with 2 anomalous points; earliest anomaly at 2021-03-23 18:01+08:00. Window mean ~258.45 (31 samples).\n"
    "- Tomcat03: log.row_count exceeded its global P95 (343.5) with 2 anomalous points; earliest anomaly at 2021-03-23 18:03+08:00. Window mean ~264.55 (31 samples).\n"
    "- Tomcat02: log.gc_pause_max exceeded its global P95 (0.0945) with 1 anomalous point; earliest anomaly at 2021-03-23 18:06+08:00 (12 samples, mean ~0.0658).\n"
    "- Tomcat02: log.row_count exceeded its global P95 (320.5) with 2 anomalous points; earliest anomaly at 2021-03-23 18:16+08:00. Window mean ~254.87 (31 samples).\n"
    "- Tomcat03: log.gc_pause_max exceeded its global P95 (0.0980) with 1 anomalous point; earliest anomaly at 2021-03-23 18:17+08:00 (12 samples, mean ~0.0625).\n\n"
    "Deduplicated cmdb_ids with log anomalies: Tomcat02, Tomcat03, Tomcat04.\n\n"
    "Interpretation / next steps:\n"
    "- There were short-lived spikes in request/row counts on Tomcat02/03/04 early in the window (18:01–18:16), and single GC pause breaches on Tomcat02/03 around 18:06–18:17.\n"
    "- Investigate Tomcat access patterns and GC activity at the listed timestamps (18:01, 18:03, 18:06, 18:16, 18:17) to see if traffic surges triggered GC or processing delays.\n"
    "- Correlate these log anomalies with the metric and trace findings (row_count and latency spikes) to pinpoint whether load or JVM GC contributed to degraded behavior."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id          log_name  global_p95  window_sample_count  count_anomalies earliest_anomaly_timestamp  window_min_value  window_max_value  window_mean
8  Tomcat04     log.row_count    325.5000                   31                2  2021-03-23 18:01:00+08:00              4.00            337.00   258.451613
5  Tomcat03     log.row_count    343.5000                   31                2  2021-03-23 18:03:00+08:00              2.00            367.00   264.548387
1  Tomcat02  log.gc_pause_max      0.0945                   12                1  2021-03-23 18:06:00+08:00              0.02              0.10     0.065833
2  Tomcat02     log.row_count    320.5000                   31                2  2021-03-23 18:16:00+08:00              3.00            348.00   254.870968
4  Tomcat03  log.gc_pause_max      0.0980                   12                1  2021-03-23 18:17:00+08:00              0.02              0.12     0.062500, 0    Tomcat02
1    Tomcat03
2    Tomcat04
Name: cmdb_id, dtype: object)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Candidates and incident window
candidates = ['IG02','MG02','MG01','Tomcat02','Tomcat03','Tomcat04']
start = tz.localize(pd.to_datetime('2021-03-23 18:00:00'))
end = tz.localize(pd.to_datetime('2021-03-23 18:30:00'))

# Read error logs
df_err = pd.read_csv('error_logs.csv')

# Ensure numeric timestamp and convert to timezone-aware datetimes
df_err['timestamp'] = pd.to_numeric(df_err['timestamp'], errors='coerce')
df_err['ts_dt'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter by candidates and incident window (inclusive)
df_filtered = df_err[
    df_err['cmdb_id'].isin(candidates) &
    (df_err['ts_dt'] >= start) &
    (df_err['ts_dt'] <= end)
].copy()

# Select and sort columns as requested, limit to 50 rows
if df_filtered.empty:
    result = []  # explicit empty list indicator
else:
    result = df_filtered.sort_values(by='ts_dt', ascending=True)[['ts_dt','cmdb_id','message']].head(50)
    # Rename ts_dt back to 'timestamp' for output column name consistency
    result = result.rename(columns={'ts_dt': 'timestamp'}) 

result

```
Out[5]:
```
summary = "No error_log entries were found for IG02, MG02, MG01, Tomcat02, Tomcat03, or Tomcat04 within 2021-03-23 18:00:00 to 2021-03-23 18:30:00 (Asia/Shanghai). Result: empty list."
summary

The original code execution output of IPython Kernel is also provided below for reference:

[]```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Candidates and incident window
candidates = ['IG02','MG02','MG01','Tomcat02','Tomcat03','Tomcat04']
start = tz.localize(pd.to_datetime('2021-03-23 18:00:00'))
end = tz.localize(pd.to_datetime('2021-03-23 18:30:00'))

# Load data
df_metric = pd.read_csv('metric.csv').rename(columns={'kpi_name': 'kpi_name'})
df_metric['source_file'] = 'metric.csv'
df_trace = pd.read_csv('trace.csv').rename(columns={'trace_name': 'kpi_name'})
df_trace['source_file'] = 'trace.csv'
df_log = pd.read_csv('log.csv').rename(columns={'log_name': 'kpi_name'})
df_log['source_file'] = 'log.csv'

# Concatenate and keep only candidates
df_all = pd.concat([df_metric, df_trace, df_log], ignore_index=True, sort=False)
df_all = df_all[df_all['cmdb_id'].isin(candidates)].copy()

# Normalize types and timestamps
df_all['value'] = pd.to_numeric(df_all['value'], errors='coerce')
df_all['timestamp'] = pd.to_numeric(df_all['timestamp'], errors='coerce')
df_all['ts_dt'] = pd.to_datetime(df_all['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Group keys
keys = ['source_file','cmdb_id','kpi_name']

def detect_breaches(df, q):
    # global quantile per group on full series
    thr = df.groupby(keys, as_index=False)['value'].quantile(q).rename(columns={'value':'global_thr'})
    # windowed points
    window_df = df[(df['ts_dt'] >= start) & (df['ts_dt'] <= end)].copy()
    if window_df.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','source_file','kpi_name','global_thr','breach_timestamp','breach_value',
            'window_sample_count','count_anomalies'
        ])
    # window sample counts per group
    window_counts = window_df.groupby(keys, as_index=False).agg(window_sample_count=('value','count'))
    # attach threshold to each window point
    window_with_thr = window_df.merge(thr, on=keys, how='left')
    # anomalies: value > global_thr
    anom_points = window_with_thr[window_with_thr['value'] > window_with_thr['global_thr']].copy()
    if anom_points.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','source_file','kpi_name','global_thr','breach_timestamp','breach_value',
            'window_sample_count','count_anomalies'
        ])
    # per-group anomaly count and earliest anomaly timestamp
    anom_agg = anom_points.groupby(keys, as_index=False).agg(
        count_anomalies=('value','count'),
        breach_timestamp=('ts_dt','min')
    )
    # breach_value: value at earliest anomaly (take first occurrence)
    anom_points_sorted = anom_points.sort_values(keys + ['ts_dt'])
    breach_first = anom_points_sorted.groupby(keys, as_index=False).first()[keys + ['ts_dt','value']].rename(
        columns={'ts_dt':'breach_timestamp','value':'breach_value'}
    )
    # combine aggregates
    combined = anom_agg.merge(breach_first, on=keys + ['breach_timestamp'], how='left')
    combined = combined.merge(thr, on=keys, how='left')
    combined = combined.merge(window_counts, on=keys, how='left')
    # select and order columns
    combined = combined[['cmdb_id','source_file','kpi_name','global_thr','breach_timestamp','breach_value','window_sample_count','count_anomalies']]
    return combined

# Try P95 first
breaches = detect_breaches(df_all, 0.95)
threshold_used = 'P95'
if breaches.empty:
    breaches = detect_breaches(df_all, 0.90)
    threshold_used = 'P90' if not breaches.empty else 'NONE'

if breaches.empty:
    final = pd.DataFrame(columns=['cmdb_id','source_file','kpi_name','global_p95','breach_timestamp','breach_value','window_sample_count','count_anomalies'])
else:
    # rename threshold column to global_p95 for output label (even if P90 used)
    breaches = breaches.rename(columns={'global_thr':'global_p95'})
    # For each component select single earliest breach across all source_files/KPIs
    breaches['breach_timestamp'] = pd.to_datetime(breaches['breach_timestamp'])
    earliest_per_component = breaches.sort_values(by='breach_timestamp', ascending=True).groupby('cmdb_id', as_index=False).first()
    # limit to up to one row per candidate and sort by breach_timestamp ascending
    final = earliest_per_component[['cmdb_id','source_file','kpi_name','global_p95','breach_timestamp','breach_value','window_sample_count','count_anomalies']].sort_values(by='breach_timestamp').head(6).reset_index(drop=True)

# Display final results and which threshold was used
final, threshold_used

```
Out[6]:
```
Threshold used: global P95.

Earliest breach per component within 2021-03-23 18:00–18:30 (Asia/Shanghai):

- IG02 — trace.csv, trace.to_Tomcat03.row_count: global P95 = 173.55; earliest breach 2021-03-23 18:00:00+08:00 with value 174.0; window_sample_count = 30; count_anomalies = 2.

- MG01 — metric.csv, OSLinux-CPU_CPU_CPUWio: global P95 = 0.841662; earliest breach 2021-03-23 18:00:00+08:00 with value 1.8969; window_sample_count = 30; count_anomalies = 2.

- MG02 — metric.csv, OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...: global P95 = 2651.0; earliest breach 2021-03-23 18:00:00+08:00 with value 2678.0; window_sample_count = 6; count_anomalies = 1.

- Tomcat02 — metric.csv, OSLinux-CPU_CPU_CPUUserTime: global P95 = 25.948367; earliest breach 2021-03-23 18:00:00+08:00 with value 26.223849; window_sample_count = 30; count_anomalies = 2.

- Tomcat03 — trace.csv, trace.to_MG01.row_count: global P95 = 167.5; earliest breach 2021-03-23 18:00:00+08:00 with value 168.0; window_sample_count = 31; count_anomalies = 2.

- Tomcat04 — metric.csv, OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...: global P95 = 0.325; earliest breach 2021-03-23 18:00:00+08:00 with value 0.4; window_sample_count = 16; count_anomalies = 1.

Interpretation: each listed component had at least one metric/trace value in the incident window exceeding its group-wide P95; the table above shows the single earliest such breach per component (with counts of anomalies observed for that KPI in-window).

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id source_file                                           kpi_name   global_p95          breach_timestamp  breach_value  window_sample_count  count_anomalies
0      IG02   trace.csv                        trace.to_Tomcat03.row_count   173.550000 2021-03-23 18:00:00+08:00    174.000000                   30                2
1      MG01  metric.csv                             OSLinux-CPU_CPU_CPUWio     0.841662 2021-03-23 18:00:00+08:00      1.896900                   30                2
2      MG02  metric.csv  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...  2651.000000 2021-03-23 18:00:00+08:00   2678.000000                    6                1
3  Tomcat02  metric.csv                        OSLinux-CPU_CPU_CPUUserTime    25.948367 2021-03-23 18:00:00+08:00     26.223849                   30                2
4  Tomcat03   trace.csv                            trace.to_MG01.row_count   167.500000 2021-03-23 18:00:00+08:00    168.000000                   31                2
5  Tomcat04  metric.csv  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...     0.325000 2021-03-23 18:00:00+08:00      0.400000                   16                1, 'P95')```
```